# ETL

In [47]:
from pathlib import Path
import pandas as pd
import numpy as np
import re
import hashlib
import html
import json
import unicodedata

In [48]:
path_parent=Path.cwd().parent/"data"/"raw"
raw_prods=pd.read_csv(path_parent/'cleaned_makeup_products.csv')
raw_prods["item_id"]=pd.to_numeric(raw_prods.item_id, errors='coerce').astype('Int64').astype('string')
raw_prods['description']=raw_prods['description'].astype('string')
raw_prods["category"]=raw_prods.category.astype('string')

## 1. Identitet proizvoda i SKU varijante

Iz URL-a se ukljanja SKU parametar i dobija kanonski proizvod, a item_id koristi kao identifikator konkretne SKU varijante.

In [49]:
raw_prods['logical_product_link']=raw_prods['product_link'].str.split('?').str[0]
raw_prods['item_id_clean']=raw_prods['product_link'].str.split('sku=').str[1]

## 2. Čišćenje atributa proizvoda

Čisti opis, izdvaja ili dopunja brend i priprema kategoriju. Izvorne verzije se čuvaju radi provera, a očišćene vrednosti koriste se u konačnoj tabeli.

### Description & Ingredients

In [ ]:
def pronadji_pocetak_sastojaka(tekst):
    posebni_format = re.search(
        r"""(?ix)
        \b(
            Ingredients\s+Active\s+Ingredients |
            Ingredients\s*/\s*Ingredients\s+Active |
            Ingredients\s+Active |
            Active\s+Ingredients
        )\s*:
        """,
        tekst
    )
    if posebni_format:
        return posebni_format.start()
    kandidati = list(re.finditer(r"(?i)\bIngredients\b\s*:?",tekst))
    for kandidat in reversed(kandidati):
        tekst_pre = tekst[max(0, kandidat.start() - 20):kandidat.start()]
        if re.search(r"(?i)\b(?:Active|Inactive)\s*$", tekst_pre):
            continue
        nastavak = tekst[kandidat.end():].strip()
        if re.match(r"(?i)^(may|are|can|could|should|subject|vary|change)\b",nastavak):
            continue
        if "," in nastavak[:500] or "%" in nastavak[:500]:
            return kandidat.start()
    return None

def razdvoji_opis_i_sastojke(opis):
    if pd.isna(opis):
        return pd.Series({"description_clean": pd.NA,"ingredients": pd.NA})
    summary = re.search(r"(?is)\bSummary\b\s*(.*)",str(opis))
    if summary is None:
        return pd.Series({
            "description_clean": pd.NA,
            "ingredients": pd.NA
        })
    tekst = summary.group(1)
    tekst = re.split(r"(?is)\bShipping\s*&\s*Coupon Restrictions\b",tekst,maxsplit=1)[0]
    pocetak = pronadji_pocetak_sastojaka(tekst)
    if pocetak is None:
        description_clean = tekst
        ingredients = pd.NA
    else:
        description_clean = tekst[:pocetak]
        ingredients = tekst[pocetak:]
    description_clean = re.sub(r"\s+"," ",description_clean).strip()

    if pd.notna(ingredients):
        ingredients = re.sub(r"\s+"," ",ingredients).strip()

    return pd.Series({"description_clean": description_clean or pd.NA,"ingredients": ingredients if pd.notna(ingredients) else pd.NA})

In [51]:
raw_prods[["description_clean", "ingredients"]] = raw_prods["description"].apply(razdvoji_opis_i_sastojke)

### Brand

In [52]:
invalid_brands = ["Ask A Question","Find your shade","Write A Review"]
brand_overrides = {"Born This Way Soft Matte Foundation": "Too Faced","ORIGINAL Liquid Mineral Concealer": "bareMinerals"}
raw_prods["product_name_position"] = raw_prods.apply(
    lambda x: (x["description"].find(x["product_name"]) if pd.notna(x["description"])and pd.notna(x["product_name"]) else -1),
    axis=1)
raw_prods["text_before_product_name"] = raw_prods.apply(
    lambda x: (x["description"][:x["product_name_position"]]if x["product_name_position"] >= 0 else pd.NA),
    axis=1)
raw_prods['brand']=raw_prods["brand"].astype("string").str.strip().replace(invalid_brands, pd.NA)
raw_prods["brand_from_description"] = raw_prods["text_before_product_name"].astype("string").str.rsplit("image", n=1).str[-1].str.replace(r"(?i)^\s*try it\s*","",regex=True).str.strip().replace("", pd.NA)
raw_prods['brand_manual_override']=raw_prods.product_name.str.strip().map(brand_overrides)

raw_prods["brand_final"] = raw_prods["brand"].combine_first(raw_prods["brand_from_description"]).combine_first(raw_prods["brand_manual_override"]).astype('string')


### Category

In [53]:
category_overrides = {
    "https://www.ulta.com/p/halo-sculpt-glow-face-palette-with-vitamin-e-pimprod2042910": "Contouring",
    "https://www.ulta.com/p/mini-cc-cream-with-spf-50-pimprod2013015": "BB & CC Creams",
    'https://www.ulta.com/p/dior-forever-fluid-skin-glow-foundation-pimprod2036824':'Foundation',
    'https://www.ulta.com/p/barepro-24hr-wear-skin-perfecting-matte-liquid-foundation-mineral-spf-20-pimprod2043221':'Foundation',
    'https://www.ulta.com/p/futurist-hydra-rescue-moisturizing-foundation-spf-45-pimprod2013152':"Foundation",
    'https://www.ulta.com/p/trick-treat-cc-active-propolis-color-correcting-cream-with-broad-spectrum-spf-45-pimprod2005371':'Tinted Moisturizer',
    'https://www.ulta.com/p/complexion-rescue-natural-matte-tinted-moisturizer-mineral-spf-30-pimprod2037151':'Tinted Moisturizer',
    'https://www.ulta.com/p/tinted-moisturizer-oil-free-natural-skin-perfector-broad-spectrum-spf-20-pimprod2025045':'Tinted Moisturizer',
    'https://www.ulta.com/p/mini-tinted-moisturizer-natural-skin-perfector-broad-spectrum-spf-30-pimprod2039349':'Tinted Moisturizer',
    'https://www.ulta.com/p/mini-tinted-moisturizer-oil-free-natural-skin-perfector-broad-spectrum-spf-20-pimprod2039444':'Tinted Moisturizer',
    'https://www.ulta.com/p/one-step-correct-brightening-correcting-primer-xlsImpprod2390219':'Face Primer',
    'https://www.ulta.com/p/buttermelt-pressed-powder-blush-pimprod2045333':'Blush',
    'https://www.ulta.com/p/smashbox-x-becca-under-eye-brightening-corrector-pimprod2028907':'Concealer',
     'https://www.ulta.com/p/pro-collagen-cleansing-balm-xlsImpprod18731145':'Makeup Remover',
    'https://www.ulta.com/p/travel-size-pro-collagen-cleansing-balm-pimprod2004741':'Makeup Remover',
    'https://www.ulta.com/p/superfood-aha-glow-cleansing-butter-pimprod2021098':'Makeup Remover',
    'https://www.ulta.com/p/pure-plush-gentle-deep-cleansing-foam-xlsImpprod13481005':'Makeup Remover',
    'https://www.ulta.com/p/take-day-off-charcoal-cleansing-balm-makeup-remover-pimprod2036304':'Makeup Remover'
}

raw_prods["category_final"] =(raw_prods["logical_product_link"].map(category_overrides)).combine_first(raw_prods["category"])


## 3. Tabela proizvoda

Objedinjuje izvorne redove u products_logical, sa tačno jednim redom po kanonskom URL-u

In [54]:
products_logical=raw_prods.groupby(by='logical_product_link').agg(
    product_name=('product_name','first'),
    brand=('brand_final', 'first'),
    category=('category_final', 'first'),
    price=('price', 'first'),
    description=('description_clean','first'),
    ingredients=("ingredients", "first"),
    pros=("pros", "first"),
    cons=("cons", "first"),
    best_uses=("best_uses", "first"),
    page_rating=("rating", "first"),
    page_review_count=("num_reviews", "first"),
)
products_logical = products_logical.sort_values(by="logical_product_link").reset_index()
products_logical["logical_product_id"] = ("P" + (products_logical.index + 1).astype(str).str.zfill(4))

### Normalizacija teksta

In [55]:
def clean_text(value):
    if pd.isna(value):
        return pd.NA
    text = html.unescape(str(value))
    text = unicodedata.normalize("NFKC", text)
    return re.sub(r"\s+", " ", text).strip()

text_columns = ["product_name","brand","category","description","pros","cons","best_uses",]
products_logical[text_columns] = products_logical[text_columns].map(clean_text).astype("string")

## 4. Dokumenti proizvoda

Od naziva, brenda, kategorije i  opisa pravi jedan  document_text po proizvodu.

In [56]:
document_columns = ["logical_product_id","product_name","brand","category","description","pros", "cons", "best_uses"]
product_documents=products_logical.loc[:,document_columns].copy()
product_documents[text_columns] = product_documents[text_columns].fillna("")

def napravi_document_text(red):
    polja = {
        "Name": red["product_name"],
        "Brand": red["brand"],
        "Category": red["category"],
        "Description": red["description"],
        "Pros": red["pros"],
        "Cons": red["cons"],
        "Best uses": red["best_uses"],
    }
    return "\n".join(f"{naziv}: {vrednost}" for naziv, vrednost in polja.items())

product_documents["document_text"] = product_documents.apply(napravi_document_text,axis=1)

In [57]:
assert products_logical.logical_product_link.is_unique
assert products_logical.logical_product_id.is_unique
assert products_logical.product_name.notna().all()
assert products_logical.brand.notna().all()
assert products_logical.category.notna().all()
for col in ["product_name", "brand", "category"]:
    assert products_logical[col].str.strip().ne("").all()

assert products_logical.price.ge(0).all()
assert products_logical.page_rating.dropna().between(0, 5).all()
assert products_logical.page_review_count.dropna().ge(0).all()

assert product_documents.document_text.notna().all()
assert product_documents.document_text.str.strip().ne("").all()

## 5. Čuvanje izlaza

In [58]:
output_dir = Path.cwd().parent / "data" / "processed"
output_dir.mkdir(parents=True, exist_ok=True)


output_tables = {
    "products_logical.csv": products_logical,
    "product_documents.csv" :product_documents,
}

for filename, dataframe in output_tables.items():
    dataframe.to_csv(output_dir / filename, index=False, encoding="utf-8")
  